In [ ]:
# WORKING GROQ MODELS (as of March 2026)
# llama-3.1-8b-instant   — fast, free, use this as default
# llama-3.3-70b-versatile — bigger, smarter, slower (use for complex tasks)
# mixtral-8x7b-32768      — alternative if llama has issues

In [ ]:
# 🤖 AI-Powered PDF Chatbot — Research Grade
## Capstone Project | RAG Pipeline

**Tech Stack:** LangChain · ChromaDB · Groq (Llama-3.1) · RAGAS · Streamlit

**Pipelines:**
- Pipeline 1: Document Ingestion
- Pipeline 2: Query & Response
- Pipeline 3: Evaluation & Experiments

In [ ]:
# ============================================================
# CELL 1 — Install Dependencies
# ============================================================
import subprocess
import sys

packages = [
    "pypdf2",
    "langchain",
    "langchain-community",
    "sentence-transformers",
    "chromadb",
    "groq",
    "pdfplumber",
    "rank-bm25",
    "ragas"
]

print("Installing packages...\n")
for package in packages:
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", package, "-q", "--no-warn-conflicts"],
        capture_output=True,
        text=True
    )
    if result.returncode == 0:
        print(f"  ✅ {package}")
    else:
        print(f"  ❌ {package} — {result.stderr.strip()}")

print("\nAll packages processed!")

In [ ]:
# ============================================================
# CELL 2 — Verify All Imports
# ============================================================
try:
    import PyPDF2
    import langchain
    import chromadb
    import pdfplumber
    from sentence_transformers import SentenceTransformer
    from rank_bm25 import BM25Okapi
    from groq import Groq
    print("✅ All libraries imported successfully!")
    print("✅ Environment is ready to build!")
except ImportError as e:
    print(f"❌ Missing library: {e}")
    print("Re-run Cell 1 to fix")

In [ ]:
# ============================================================
# CELL 3 — Connect to Groq (Llama-3.1)
# ============================================================
from google.colab import userdata
from groq import Groq

api_key = userdata.get('GROQ_API_KEY')
client = Groq(api_key=api_key)

# Quick test
response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[{"role": "user", "content": "Say: Groq connection successful!"}]
)

print("✅ " + response.choices[0].message.content)



In [ ]:
# ============================================================
# CELL 4 — Upload PDF to Colab
# ============================================================
from google.colab import files

print("A file picker will appear below...")
print("Select your rag_paper.pdf file from your laptop\n")

uploaded = files.upload()

# Show what was uploaded
for filename in uploaded.keys():
    print(f"✅ Successfully uploaded: {filename}")
    print(f"   Size: {len(uploaded[filename]):,} bytes")

In [ ]:
# ============================================================
# CELL 5 — Extract Text from PDF
# ============================================================
import PyPDF2

# Open and read the PDF
pdf_file = open("rag_paper.pdf", "rb")  # rb = read binary
reader = PyPDF2.PdfReader(pdf_file)

# How many pages?
total_pages = len(reader.pages)
print(f"✅ PDF loaded successfully!")
print(f"📄 Total pages: {total_pages}")
print()

# Extract text from all pages
full_text = ""
for page_num in range(total_pages):
    page = reader.pages[page_num]
    text = page.extract_text()
    full_text += text

print(f"📝 Total characters extracted: {len(full_text):,}")
print()

# Preview first 500 characters so we can see it worked
print("--- PREVIEW OF EXTRACTED TEXT ---")
print(full_text[:500])
print("...")

In [ ]:
!pip install langchain-text-splitters -q

In [ ]:
# ============================================================
# CELL 6 — Split Text into Chunks
# ============================================================
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Create our splitter with settings from our config
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,        # each chunk = max 800 characters
    chunk_overlap=100,     # chunks share 100 characters with neighbours
                           # overlap helps so answers don't get cut off at edges
    separators=["\n\n", "\n", ".", " ", ""]  # try to split at natural breaks
)

# Split the full text into chunks
chunks = text_splitter.create_documents([full_text])

# Results
print(f"✅ Chunking complete!")
print(f"📄 Original text: {len(full_text):,} characters")
print(f"✂️  Total chunks created: {len(chunks)}")
print(f"📏 Average chunk size: {len(full_text) // len(chunks)} characters")
print()

# Preview first 3 chunks so we can see what they look like
print("--- PREVIEW: FIRST 3 CHUNKS ---")
for i, chunk in enumerate(chunks[:3]):
    print(f"\n🔹 Chunk {i+1} ({len(chunk.page_content)} chars):")
    print(chunk.page_content)
    print("-" * 50)

In [ ]:
# ============================================================
# CELL 7 — Generate Embeddings
# ============================================================
from sentence_transformers import SentenceTransformer

print("⏳ Loading embedding model (bge-small-en)...")
print("   This downloads once and is cached after that\n")

# Load our free embedding model
embedding_model = SentenceTransformer("BAAI/bge-small-en")

print("✅ Embedding model loaded!\n")

# Extract just the text from our chunks
chunk_texts = [chunk.page_content for chunk in chunks]

print(f"⏳ Generating embeddings for {len(chunk_texts)} chunks...")
print("   Converting each chunk from text → numbers\n")

# Generate embeddings for all chunks
embeddings = embedding_model.encode(
    chunk_texts,
    show_progress_bar=True,  # shows a progress bar
    batch_size=32
)

print(f"\n✅ Embeddings generated!")
print(f"📊 Shape: {embeddings.shape}")
print(f"   → {embeddings.shape[0]} chunks each converted to {embeddings.shape[1]} numbers")
print(f"\n🔍 Preview of first chunk's embedding (first 5 numbers):")
print(f"   {embeddings[0][:5]}")
print(f"\n   These numbers represent the MEANING of Chunk 1 mathematically!")

In [ ]:
# ============================================================
# CELL 8 — Store Embeddings in ChromaDB
# ============================================================
import chromadb
import uuid

print("⏳ Setting up ChromaDB vector store...\n")

# Create a ChromaDB client that stores data in memory
# (We will add persistent storage in a later phase)
chroma_client = chromadb.Client()

# Create a collection — think of this like a table in a database
# Delete if exists first (so we can re-run this cell cleanly)
try:
    chroma_client.delete_collection("rag_paper")
except:
    pass

collection = chroma_client.create_collection(
    name="rag_paper",
    metadata={"description": "RAG research paper chunks"}
)

print("✅ ChromaDB collection created!\n")

# Add all chunks and their embeddings to ChromaDB
print(f"⏳ Storing {len(chunks)} chunks in ChromaDB...")

collection.add(
    ids=[str(uuid.uuid4()) for _ in chunks],        # unique ID for each chunk
    embeddings=embeddings.tolist(),                  # the 384 numbers per chunk
    documents=chunk_texts,                           # the original text
    metadatas=[{"chunk_index": i, "source": "rag_paper.pdf"}
               for i in range(len(chunks))]          # extra info per chunk
)

print(f"✅ All chunks stored successfully!")
print(f"📦 Total items in ChromaDB: {collection.count()}")
print()

# Test it works — search for something!
print("🔍 Running a test search: 'what is retrieval augmented generation?'")
print()

test_query_embedding = embedding_model.encode(
    ["what is retrieval augmented generation?"]
).tolist()

results = collection.query(
    query_embeddings=test_query_embedding,
    n_results=3
)

print("--- TOP 3 MOST RELEVANT CHUNKS FOUND ---")
for i, doc in enumerate(results["documents"][0]):
    print(f"\n🔹 Result {i+1}:")
    print(doc[:300])
    print("...")

In [ ]:
# ============================================================
# CELL 9 — Checkpoint Save / Load
# ============================================================
import pickle
import os

checkpoint_exists = (
    os.path.exists("checkpoints/chunks.pkl") and
    os.path.exists("checkpoints/embeddings.pkl") and
    os.path.exists("checkpoints/chunk_texts.pkl")
)

if checkpoint_exists:
    # ── LOAD from checkpoint ──────────────────────────────
    print("✅ Checkpoint found! Loading saved data...\n")

    with open("checkpoints/chunks.pkl", "rb") as f:
        chunks = pickle.load(f)

    with open("checkpoints/embeddings.pkl", "rb") as f:
        embeddings = pickle.load(f)

    with open("checkpoints/chunk_texts.pkl", "rb") as f:
        chunk_texts = pickle.load(f)

    print(f"✅ Loaded {len(chunks)} chunks")
    print(f"✅ Embeddings shape: {embeddings.shape}")
    print(f"✅ Chunk texts: {len(chunk_texts)}")
    print()
    print("🚀 Ready to build! No need to re-run Cells 4-8")

else:
    # ── SAVE checkpoint ───────────────────────────────────
    print("💾 No checkpoint found — saving now...\n")

    os.makedirs("checkpoints", exist_ok=True)

    with open("checkpoints/chunks.pkl", "wb") as f:
        pickle.dump(chunks, f)

    with open("checkpoints/embeddings.pkl", "wb") as f:
        pickle.dump(embeddings, f)

    with open("checkpoints/chunk_texts.pkl", "wb") as f:
        pickle.dump(chunk_texts, f)

    print(f"✅ Checkpoint saved!")
    print(f"   → chunks.pkl     ({len(chunks)} chunks)")
    print(f"   → embeddings.pkl ({embeddings.shape})")
    print(f"   → chunk_texts.pkl ({len(chunk_texts)} texts)")
    print()
    print("💡 Next session: run Cells 1, 2, 3 then this cell only!")

In [ ]:
# ============================================================
# CELL 10 — Rebuild ChromaDB from Checkpoint
# ============================================================
import chromadb
import uuid
import logging
from sentence_transformers import SentenceTransformer

# Suppress unnecessary warnings for clean output
logging.getLogger("sentence_transformers").setLevel(logging.ERROR)
logging.getLogger("transformers").setLevel(logging.ERROR)

print("⏳ Rebuilding ChromaDB from checkpoint...\n")

# Reload embedding model
embedding_model = SentenceTransformer("BAAI/bge-small-en")
print("✅ Embedding model loaded!")

# Rebuild ChromaDB
chroma_client = chromadb.Client()

try:
    chroma_client.delete_collection("rag_paper")
except:
    pass

collection = chroma_client.create_collection(
    name="rag_paper",
    metadata={"description": "RAG research paper chunks"}
)

collection.add(
    ids=[str(uuid.uuid4()) for _ in chunks],
    embeddings=embeddings.tolist(),
    documents=chunk_texts,
    metadatas=[{"chunk_index": i, "source": "rag_paper.pdf"}
               for i in range(len(chunks))]
)

print(f"✅ ChromaDB rebuilt!")
print(f"📦 Total chunks in database: {collection.count()}")
print()
print("🚀 Pipeline 2 ready to build!")

In [ ]:
# ============================================================
# CELL 11 — Retrieval Function
# ============================================================

def retrieve_chunks(question, top_k=4):
    """
    Takes a question, searches ChromaDB, returns most relevant chunks.

    Args:
        question: The user's question as a string
        top_k: How many chunks to return (default 4)

    Returns:
        List of dicts with 'text', 'source', 'chunk_index', 'confidence'
    """

    # Step 1 — Convert question to embedding vector
    question_embedding = embedding_model.encode([question]).tolist()

    # Step 2 — Search ChromaDB for most similar chunks
    results = collection.query(
        query_embeddings=question_embedding,
        n_results=top_k,
        include=["documents", "metadatas", "distances"]
    )

    # Step 3 — Package results into clean format
    retrieved = []
    for i in range(len(results["documents"][0])):

        # Convert distance to confidence score
        # ChromaDB returns distance (lower = more similar)
        # We convert to 0-1 score (higher = more confident)
        distance = results["distances"][0][i]
        confidence = round(1 - (distance / 2), 3)

        retrieved.append({
            "text":        results["documents"][0][i],
            "source":      results["metadatas"][0][i]["source"],
            "chunk_index": results["metadatas"][0][i]["chunk_index"],
            "confidence":  confidence
        })

    return retrieved


# ── TEST IT ───────────────────────────────────────────────
print("🔍 Testing retrieval function...\n")

test_question = "How does RAG combine parametric and non-parametric memory?"
results = retrieve_chunks(test_question, top_k=4)

print(f"Question: {test_question}")
print(f"Found {len(results)} chunks\n")
print("--- RETRIEVED CHUNKS ---")

for i, chunk in enumerate(results):
    print(f"\n🔹 Chunk {i+1}")
    print(f"   Source:      {chunk['source']}")
    print(f"   Chunk index: {chunk['chunk_index']}")
    print(f"   Confidence:  {chunk['confidence']}")
    print(f"   Preview:     {chunk['text'][:200]}...")

In [ ]:
# ============================================================
# CELL 12 — Prompt Builder
# ============================================================

def build_prompt(question, retrieved_chunks):
    """
    Combines the user question and retrieved chunks into
    a structured prompt for Llama-3.1.

    Args:
        question: The user's question
        retrieved_chunks: List of dicts from retrieve_chunks()

    Returns:
        system_prompt: Instructions for the LLM
        user_prompt: Question + context combined
    """

    # Build context block from retrieved chunks
    context_blocks = []
    for i, chunk in enumerate(retrieved_chunks):
        context_blocks.append(
            f"[Source {i+1} | {chunk['source']} | "
            f"Chunk {chunk['chunk_index']} | "
            f"Confidence: {chunk['confidence']}]\n"
            f"{chunk['text']}"
        )

    context = "\n\n---\n\n".join(context_blocks)

    # System prompt — tells Llama how to behave
    system_prompt = """You are an expert AI research assistant that answers questions based strictly on provided context.

Your response MUST follow this exact format:

ANSWER:
[Your detailed answer here, based only on the provided context]

SOURCES:
[List which Source numbers you used, e.g. Source 1, Source 3]

CONFIDENCE:
[Rate your confidence as High / Medium / Low based on how well the context answers the question]

FOLLOW-UP QUESTIONS:
1. [A relevant follow-up question]
2. [Another relevant follow-up question]
3. [A third relevant follow-up question]

IMPORTANT RULES:
- Only use information from the provided context
- If the context does not contain enough information, say so clearly
- Never make up facts or hallucinate information
- Always cite which sources you used"""

    # User prompt — question + retrieved context
    user_prompt = f"""Please answer this question using the provided context:

QUESTION: {question}

CONTEXT:
{context}"""

    return system_prompt, user_prompt


# ── TEST IT ───────────────────────────────────────────────
print("🔧 Testing prompt builder...\n")

test_question = "How does RAG combine parametric and non-parametric memory?"
retrieved = retrieve_chunks(test_question, top_k=4)
system_prompt, user_prompt = build_prompt(test_question, retrieved)

print("✅ Prompt built successfully!")
print()
print(f"📏 System prompt length: {len(system_prompt)} characters")
print(f"📏 User prompt length:   {len(user_prompt)} characters")
print(f"📏 Total prompt length:  {len(system_prompt) + len(user_prompt)} characters")
print()
print("--- PREVIEW OF USER PROMPT (first 500 chars) ---")
print(user_prompt[:500])
print("...")

In [ ]:
# ============================================================
# CELL 13 — LLM Caller (Send to Llama-3.1)
# ============================================================
import time

def get_answer(question, top_k=4):
    """
    Full pipeline: retrieve → build prompt → call LLM → return answer.

    Args:
        question: The user's question
        top_k: Number of chunks to retrieve

    Returns:
        Dict with answer, sources, confidence, follow_ups, latency
    """

    start_time = time.time()

    # Step 1 — Retrieve relevant chunks
    retrieved = retrieve_chunks(question, top_k=top_k)

    # Step 2 — Build prompt
    system_prompt, user_prompt = build_prompt(question, retrieved)

    # Step 3 — Call Llama-3.1 via Groq
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt}
        ],
        temperature=0.1,   # low = more factual, less creative
        max_tokens=1000
    )

    # Step 4 — Extract and return results
    latency = round(time.time() - start_time, 2)
    answer_text = response.choices[0].message.content

    return {
        "question":  question,
        "answer":    answer_text,
        "chunks":    retrieved,
        "latency":   latency,
        "model":     "llama-3.1-8b-instant"
    }


# ── TEST IT — THE MOMENT OF TRUTH ─────────────────────────
print("🤖 Sending question to Llama-3.1...\n")
print("=" * 60)

test_question = "How does RAG combine parametric and non-parametric memory?"
result = get_answer(test_question)

print(f"❓ QUESTION: {result['question']}")
print(f"⏱️  Latency:  {result['latency']} seconds")
print(f"🤖 Model:    {result['model']}")
print()
print("=" * 60)
print("💬 ANSWER:")
print("=" * 60)
print(result["answer"])
print("=" * 60)

In [ ]:
# ============================================================
# CELL 14 — Full Pipeline Test (5 Questions)
# ============================================================

test_questions = [
    "What is retrieval augmented generation?",
    "What datasets were used to evaluate RAG?",
    "How does RAG perform on open domain question answering?",
    "What are the limitations of RAG?",
    "What is the difference between RAG-Token and RAG-Sequence?"
]

print("🧪 Running full pipeline test with 5 questions...\n")
print("=" * 60)

all_results = []

for i, question in enumerate(test_questions):
    print(f"\n❓ Question {i+1}: {question}")
    print("-" * 60)

    result = get_answer(question)
    all_results.append(result)

    # Show condensed output for each question
    lines = result["answer"].split("\n")

    # Extract just the ANSWER section
    answer_lines = []
    in_answer = False
    for line in lines:
        if line.startswith("ANSWER:"):
            in_answer = True
            continue
        if line.startswith("SOURCES:") or line.startswith("CONFIDENCE:"):
            in_answer = False
        if in_answer and line.strip():
            answer_lines.append(line)

    short_answer = " ".join(answer_lines)[:300]

    # Extract confidence
    confidence = "Unknown"
    for line in lines:
        if line.startswith("High") or line.startswith("Medium") or line.startswith("Low"):
            confidence = line.strip()
            break

    print(f"💬 Answer: {short_answer}...")
    print(f"🎯 Confidence: {confidence}")
    print(f"⏱️  Latency: {result['latency']} seconds")

print()
print("=" * 60)
print("📊 PIPELINE 2 SUMMARY")
print("=" * 60)

avg_latency = round(sum(r['latency'] for r in all_results) / len(all_results), 2)
print(f"✅ Questions answered:  {len(all_results)}")
print(f"✅ Average latency:     {avg_latency} seconds")
print(f"✅ Model used:          llama-3.1-8b-instant")
print(f"✅ Chunks retrieved:    4 per question")
print(f"✅ Total chunks used:   {len(all_results) * 4}")
print()
print("🎉 Pipeline 2 — Query & Response — COMPLETE!")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ============================================================
# TEMP — Fix & Save Clean Notebook to Drive
# (Delete this cell after GitHub is updated)
# ============================================================
import json

notebook_path = "/content/drive/MyDrive/Colab Notebooks/pdf-chatbot-capstone.ipynb"

with open(notebook_path, "r", encoding="utf-8") as f:
    nb = json.load(f)

nb["metadata"] = {}
for cell in nb.get("cells", []):
    cell["outputs"] = []
    cell["execution_count"] = None

clean_path = "/content/pdf-chatbot-capstone-clean.ipynb"
with open(clean_path, "w", encoding="utf-8") as f:
    json.dump(nb, f, indent=1)

print("✅ Clean notebook ready!")
print()
print("Now:")
print("1. Click 📁 folder icon on LEFT sidebar")
print("2. Find pdf-chatbot-capstone-clean.ipynb")
print("3. Right click → Download")
print("4. Go to GitHub → notebooks folder")
print("5. Click existing file → click pencil edit icon → ...")
print("   OR Add file → Upload files → upload and overwrite")